In [1]:
import os
import re

import numpy as np
import torch
import pandas as pd
import matplotlib
import seaborn as sns
import pickle
from tqdm.notebook import tqdm
from functionsgpu_old import *
from plotting_betas import *
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from video_saving import *

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)
dtype = torch.float32
SEED = 42

def deterministic():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Enable deterministic operations (may slow down training slightly)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

deterministic()
tslen = 200

functionsgpu_old.py device: cuda:1
cuda:1


## Load POMA and Participant IDs

In [2]:
# Load POMA scores and participant IDs (same logic as in fpca.ipynb)
import re

folder_path = "/mnt/sdb/arafat/stroke_riemann/csv_r"
files = [file for file in os.listdir(folder_path)]
files = sorted(files, key=lambda x: int(x.split('_')[0][2:]))

scores_csv = [file.split('_')[1] for file in files]
scores = [file.split('.')[0] for file in scores_csv]

y_poma = np.array(scores).astype(int)

# Participant ID for each row of dataset (same order as files from csv_r)
participant_ids = [re.search(r'ID(\d+)_', f).group(1) for f in files]
step = 10  # leave-10-participants-out per fold

print("y_poma shape:", y_poma.shape)
print("First 10 participant_ids:", participant_ids[:10])

y_poma shape: (155,)
First 10 participant_ids: ['1', '2', '3', '4', '5', '6', '7', '8', '10', '11']


## Data Loading

In [3]:
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all

betas_aligned_all, mu_all_t, tangent_vec_all = loading('aligned_data',tslen)
mu_all_t_tensor = torch.from_numpy(mu_all_t).to(device=device, dtype=torch.float32)
betas_aligned = np.array(betas_aligned_all)
betas_aligned = betas_aligned.transpose(1, 2, 3, 0)
print(betas_aligned.shape, tangent_vec_all.shape, mu_all_t.shape)

(32, 3, 200, 155) (32, 3, 200, 155) (32, 3, 200)


In [4]:
K = 32
M = 3
T = tslen
nsamples = 155

tangent_flat = tangent_vec_all.reshape((K*M*T, nsamples))
print(tangent_flat.shape)

(19200, 155)


# Nonlinear Tangent VAE

In [5]:
class NonlinearVAE(nn.Module):
    """NonlinearVAE"""
    def __init__(self, D, R, H=64, dropout=0.1):
        super().__init__()
        
        # Encoder layers
        self.W1 = nn.Linear(D, H, bias=False)        # input -> hidden
        self.W2_mu = nn.Linear(H, R, bias=False)     # hidden -> latent mean
        self.W2_logvar = nn.Linear(H, R)             # hidden -> latent logvar
        self.dropout = nn.Dropout(p=dropout)
        
        # Decoder layers
        self.dec1 = nn.Linear(R, 16, bias=False)
        self.dec2 = nn.Linear(16, D, bias=False)

    def encode(self, x):
        h = torch.tanh(self.W1(x))
        h = self.dropout(h)
        mu = self.W2_mu(h)
        logvar = self.W2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h_recon = torch.tanh(self.dec1(z))
        x_hat = self.dec2(h_recon)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def vae_loss(x, x_hat, mu, logvar, beta=1e-4):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
    return recon + beta * kl.mean(), recon, kl.mean()


## Training Function for Each Fold Training (VAE)

In [9]:
def train_vae_fold(X_tan_train, D, R, num_epochs=1000, lr=1e-3, verbose=False, seed=42, batch_size=32):
    """Train a fresh NonlinearVAE on a training subset only (tangent space MSE + KL loss).

    X_tan_train : (N_train, D) tangent vectors
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    # Batched loader similar to Nonlinear_AE_VAE_RVAE
    dataset_fold = TensorDataset(X_tan_train)
    g_fold = torch.Generator(device=device).manual_seed(seed)
    loader_fold = DataLoader(dataset_fold, batch_size=batch_size, shuffle=True, drop_last=False, generator=g_fold)

    model_fold.train()
    for epoch in range(num_epochs):
        epoch_loss, epoch_recon, epoch_kl, num_samples = 0.0, 0.0, 0.0, 0

        for (x_batch,) in loader_fold:
            x_batch = x_batch.to(device)

            opt_fold.zero_grad(set_to_none=True)
            x_hat, mu, logvar, z = model_fold(x_batch)
            loss_train, recon_train, kl_train = vae_loss(x_batch, x_hat, mu, logvar, beta=1e-5)
            loss_train.backward()
            opt_fold.step()

            bs = x_batch.size(0)
            epoch_loss += loss_train.item() * bs
            epoch_recon += recon_train.item() * bs
            epoch_kl += kl_train.item() * bs
            num_samples += bs

        if verbose and (epoch % 300 == 0 or epoch == num_epochs - 1):
            avg_loss = epoch_loss / num_samples
            avg_recon = epoch_recon / num_samples
            avg_kl = epoch_kl / num_samples
            print(f"[fold VAE] epoch {epoch} | loss {avg_loss:.6f} | recon {avg_recon:.6f} | kl {avg_kl:.6f}")
    model_fold.eval()
    return model_fold

## VAE Cross Validation

In [ ]:
# VAE latent regression: same CV as RVAE (5 val + 5 test per fold, two rounds)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from val_test import val_test
from print_results import print_results

dtype = torch.float32
n = len(y_poma)
D = tangent_flat.shape[0]
n_folds = 30
R_vae = 38

models = {
    'KNN': KNeighborsRegressor(),
    'SVM': SVR(kernel='rbf'),
    'RF': RandomForestRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42),
    'MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}
all_results_validation_vae = {name: {'targets': [], 'preds': []} for name in models.keys()}
all_results_test_vae = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)


for k in tqdm(range(n_folds), total=n_folds, desc='VAE folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
        continue

    fold_seed = SEED + k
    X_tan_train = torch.from_numpy(tangent_flat[:, train_idx].T.astype(np.float32)).to(device=device, dtype=dtype)
    model_fold = train_vae_fold(X_tan_train, D, R_vae, num_epochs=2000, lr=1e-3, verbose=True, seed=fold_seed)

    with torch.no_grad():
        mu_train_fold, _ = model_fold.encode(X_tan_train)
        mu_val_fold, _ = model_fold.encode(torch.from_numpy(tangent_flat[:, validation_idx].T.astype(np.float32)).to(device=device, dtype=dtype))
        mu_test_fold, _ = model_fold.encode(torch.from_numpy(tangent_flat[:, test_idx].T.astype(np.float32)).to(device=device, dtype=dtype))

    Z_train_fold = mu_train_fold.cpu().numpy()
    Z_val_fold = mu_val_fold.cpu().numpy()
    Z_test_fold = mu_test_fold.cpu().numpy()
    y_train_fold = y_poma[train_idx]
    y_val_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    for name, model_reg in models.items():
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)
        validation_preds = m.predict(Z_val_fold)
        test_preds = m.predict(Z_test_fold)
        all_results_validation_vae[name]['targets'].extend(y_val_fold.tolist())
        all_results_validation_vae[name]['preds'].extend(validation_preds.tolist())
        all_results_test_vae[name]['targets'].extend(y_test_fold.tolist())
        all_results_test_vae[name]['preds'].extend(test_preds.tolist())
        all_results_test_vae[name]['subjects'].extend(participant_ids[test_idx].tolist())
        mae_val = mean_absolute_error(y_val_fold, validation_preds)
        rmse_val = np.sqrt(mean_squared_error(y_val_fold, validation_preds))
        r2_val = r2_score(y_val_fold, validation_preds)
        print(f"Fold {k + 1:02d} | {name} | Validation: MAE={mae_val:.3f}, RMSE={rmse_val:.3f}, R2={r2_val:.3f}")

results_test_vae_df = print_results(all_results_validation_vae, all_results_test_vae, models)
results_test_vae_df

VAE folds:   0%|          | 0/30 [00:00<?, ?it/s]

[fold VAE] epoch 0 | loss 0.000268 | recon 0.000268 | kl 0.001727
[fold VAE] epoch 300 | loss 0.000123 | recon 0.000123 | kl 0.021788
[fold VAE] epoch 600 | loss 0.000109 | recon 0.000109 | kl 0.046379
[fold VAE] epoch 900 | loss 0.000098 | recon 0.000097 | kl 0.087506
[fold VAE] epoch 1200 | loss 0.000095 | recon 0.000094 | kl 0.104181
[fold VAE] epoch 1500 | loss 0.000090 | recon 0.000089 | kl 0.130383
[fold VAE] epoch 1800 | loss 0.000087 | recon 0.000086 | kl 0.143709
[fold VAE] epoch 1999 | loss 0.000086 | recon 0.000085 | kl 0.148944
Fold 01 | KNN | Validation: MAE=2.600, RMSE=3.680, R2=0.436
Fold 01 | SVM | Validation: MAE=2.503, RMSE=3.036, R2=0.616
Fold 01 | RF | Validation: MAE=2.352, RMSE=2.760, R2=0.683
Fold 01 | XGBoost | Validation: MAE=3.098, RMSE=3.519, R2=0.484
Fold 01 | MLP | Validation: MAE=2.397, RMSE=3.326, R2=0.539
[fold VAE] epoch 0 | loss 0.000266 | recon 0.000266 | kl 0.002266
[fold VAE] epoch 300 | loss 0.000118 | recon 0.000118 | kl 0.028561
[fold VAE] epoch 

,MAE,RMSE,R2,Pearson r,Pearson p
KNN,1.330323,2.893775,0.725796,0.853020,4.622594e-45
SVM,1.515977,3.208289,0.662952,0.847629,5.856798e-44
RF,1.469484,3.230743,0.658218,0.817104,1.978933e-38
XGBoost,1.608256,3.625975,0.569479,0.785859,9.477359e-34
MLP,1.666291,3.194038,0.665940,0.818458,1.184676e-38


In [10]:
from ci import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_test_vae[name]['targets'],
    all_results_test_vae[name]['preds'],
    all_results_test_vae[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,1.330323,2.893775,0.725796,0.85302
ci,"[1.018, 1.635]","[2.339, 3.308]","[0.647, 0.803]","[0.809, 0.9]"


# KT-RSV

In [6]:
class KendallRVAE(nn.Module):
    """rVAE: encode tangent vectors, decode to tangent, but (during
    training) compare reconstructions on the manifold via an exp map.

    - Inputs: tangent vectors (N, D) as in the VAE section.
    - Decoder: produces tangent vectors.
    - Training loss: uses expmap(mu, v_hat) vs. original manifold trajectory.
    """
    def __init__(self, base_vae, mu_shape, expmap):
        super().__init__()
        self.vae = base_vae
        # mean shape, used for exponential map when mapping back to manifold
        self.register_buffer("mu_shape", mu_shape)
        self.expmap = expmap

    def forward(self, x):
        """Forward on tangent vectors.

        x : (N, D) tangent vectors
        Returns
        -------
        x_man_hat : (N, D) manifold trajectory flattened (via expmap)
        mu_z, logvar, z, v_hat : usual VAE outputs (in tangent space)
        """
        v_hat, mu_z, logvar, z = self.vae(x)   # tangent reconstruction

        # Map reconstructed tangent field back to the manifold
        B = v_hat.shape[0]
        v_hat_reshaped = v_hat.view(B, K, M, T)
        mu = self.mu_shape.view(K, M, T)
        x_recon_man = self.expmap(mu, v_hat_reshaped)   # (B, K, M, T)
        x_recon_man = x_recon_man.view(B, -1)

        return x_recon_man, mu_z, logvar, z, v_hat

## Getting Orginal Trajectories on Manifold

In [5]:
betas = np.array(betas_aligned_all)
print(betas.shape)   # (155, 32, 3, 200)

N, K, M, T = betas.shape   # correct order

# Shape mean as before (kept for potential manifold utilities)
mu_shape = torch.from_numpy(
    mu_all_t.reshape(-1).astype(np.float32)
).to(device)

print("mu_shape:", mu_shape.shape)      # should be (19200,)

(155, 32, 3, 200)
mu_shape: torch.Size([19200])


## Training on Full Dataset (test run)

In [13]:
deterministic()
# Tangent input (as in the VAE section)
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
mu_shape = mu_shape.to(device=device, dtype=dtype)

# Corresponding manifold trajectories (flattened)
betas = np.array(betas_aligned_all)              # (N, K, M, T)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

D = X_tan.shape[1]
R = 38
kl_beta = 2**(-4)  # match Nonlinear_AE_VAE_RVAE

# Dataset and DataLoader for batched training (similar to Nonlinear_AE_VAE_RVAE)
dataset_ktrsv = TensorDataset(X_tan, X_man)
g_ktrsv = torch.Generator(device=device).manual_seed(SEED)
loader_ktrsv = DataLoader(dataset_ktrsv, batch_size=32, shuffle=True, drop_last=False, generator=g_ktrsv)

base_vae = NonlinearVAE(D, R).to(device=device, dtype=dtype)
model = KendallRVAE(base_vae, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

elbo_losses = []
avgrecon_losses = []
avgkl_losses = []

for epoch in range(3000):
    epoch_elbo, epoch_recon, epoch_kl, num_samples = 0.0, 0.0, 0.0, 0

    for x_tan_batch, x_man_batch in loader_ktrsv:
        x_tan_batch = x_tan_batch.to(device)
        x_man_batch = x_man_batch.to(device)

        opt.zero_grad(set_to_none=True)

        # forward in tangent space, loss in manifold space
        x_hat_man, mu, logvar, z, v_hat = model(x_tan_batch)

        # Geodesic reconstruction loss on the manifold
        dist = squared_geodesic_distance(x_man_batch, x_hat_man, K, M, T)
        avgrecon = dist.sum(dim=1).mean()
        kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
        avgkl = kl.mean()

        elbo_loss = avgrecon + kl_beta * avgkl

        elbo_loss.backward()
        opt.step()

        batch_size = x_tan_batch.size(0)
        epoch_elbo += elbo_loss.item() * batch_size
        epoch_recon += avgrecon.item() * batch_size
        epoch_kl += avgkl.item() * batch_size
        num_samples += batch_size

    elbo_losses.append(epoch_elbo / num_samples)
    avgrecon_losses.append(epoch_recon / num_samples)
    avgkl_losses.append(epoch_kl / num_samples)

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | ELBO {elbo_losses[-1]:.6f} | recon {avgrecon_losses[-1]:.6f} | KL {avgkl_losses[-1]:.6f}")

Epoch 0 | ELBO 1147.740855 | recon 1147.730257 | KL 0.169565
Epoch 200 | ELBO 2.363103 | recon 2.255221 | KL 1.726103
Epoch 400 | ELBO 2.216135 | recon 2.079320 | KL 2.189046
Epoch 600 | ELBO 2.195864 | recon 2.053318 | KL 2.280751
Epoch 800 | ELBO 2.058127 | recon 1.879829 | KL 2.852778
Epoch 1000 | ELBO 2.028826 | recon 1.824665 | KL 3.266573
Epoch 1200 | ELBO 1.926105 | recon 1.720557 | KL 3.288754
Epoch 1400 | ELBO 1.886138 | recon 1.688403 | KL 3.163772
Epoch 1600 | ELBO 1.836314 | recon 1.626480 | KL 3.357343
Epoch 1800 | ELBO 1.822845 | recon 1.605800 | KL 3.472725
Epoch 2000 | ELBO 1.822118 | recon 1.601271 | KL 3.533544
Epoch 2200 | ELBO 1.849269 | recon 1.633880 | KL 3.446228
Epoch 2400 | ELBO 1.817859 | recon 1.604549 | KL 3.412969
Epoch 2600 | ELBO 1.818873 | recon 1.603813 | KL 3.440962
Epoch 2800 | ELBO 1.816980 | recon 1.596123 | KL 3.533715


In [14]:
%matplotlib qt5
plot_one_traj(x_hat_man[1,:].reshape(32,3,tslen))

libGL error: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: swrast


## Training Function for Each Fold Training (KT-RSV)

In [15]:
def train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T, R=10,
                             num_epochs=2000, lr=1e-3, verbose=False, kl_beta=2**(-4), batch_size=32, seed=SEED):
    """Train a fresh KendallRVAE on a training subset only.

    X_tan_train : (N_train, D) tangent vectors
    X_man_train : (N_train, K*M*T) manifold trajectories (flattened)
    seed : random seed for model initialization
    """
    deterministic()
    D = X_tan_train.shape[1]

    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = KendallRVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    # Batched loader similar to full KTRSV training
    dataset_fold = TensorDataset(X_tan_train, X_man_train)
    g_fold = torch.Generator(device=device).manual_seed(seed)
    loader_fold = DataLoader(dataset_fold, batch_size=batch_size, shuffle=True, drop_last=False, generator=g_fold)

    model_fold.train()
    for epoch in range(num_epochs):
        epoch_elbo, epoch_recon, epoch_kl, num_samples = 0.0, 0.0, 0.0, 0

        for x_tan_batch, x_man_batch in loader_fold:
            x_tan_batch = x_tan_batch.to(device)
            x_man_batch = x_man_batch.to(device)

            opt_fold.zero_grad(set_to_none=True)

            x_hat_man_train, mu_train, logvar_train, z_train, v_hat_train = model_fold(x_tan_batch)

            dist_train = squared_geodesic_distance(x_man_batch, x_hat_man_train, K, M, T)
            avgrecon_train = dist_train.sum(dim=1).mean()
            kl_per_sample = -0.5 * (1 + logvar_train - mu_train.pow(2) - logvar_train.exp()).sum(dim=1)
            avgkl_train = kl_per_sample.mean()

            elbo_train = avgrecon_train + kl_beta * avgkl_train

            elbo_train.backward()
            opt_fold.step()

            bs = x_tan_batch.size(0)
            epoch_elbo += elbo_train.item() * bs
            epoch_recon += avgrecon_train.item() * bs
            epoch_kl += avgkl_train.item() * bs
            num_samples += bs

        if verbose and (epoch % 300 == 0 or epoch == num_epochs - 1):
            avg_loss = epoch_elbo / num_samples
            avg_recon = epoch_recon / num_samples
            avg_kl = epoch_kl / num_samples
            print(f"[fold RVAE] epoch {epoch} | loss {avg_loss:.6f} | recon {avg_recon:.6f} | kl {avg_kl:.6f}")

    model_fold.eval()
    return model_fold

## KT-RSV Cross Validation

In [ ]:
# Nested CV: re-train KendallRVAE inside each participant fold

# Set random seeds for reproducibility
import random
import numpy as np
import torch

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from val_test import val_test
from print_results import print_results
from tqdm.notebook import tqdm


# Regression models with random_state set for reproducibility
models = {
    'KNN': KNeighborsRegressor(),
    'SVM': SVR(kernel='rbf'),  # SVR doesn't have random_state parameter
    'RF': RandomForestRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42),
    'MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_poma)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 30
R = 38

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)

    # Slice tangent and manifold data for this fold
    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]

    # Train fold-specific KendallRVAE on train participants only
    model_fold = train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T,
                                         R=R, num_epochs=2000, lr=1e-3, verbose=True, seed=fold_seed)

    # Extract latent means (mu) for train, validation, and test using the fold-specific encoder
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_validation_fold, _ = model_fold.vae.encode(X_tan[validation_idx])
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_validation_fold = mu_validation_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    y_train_fold = y_poma[train_idx]
    y_validation_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold validation metrics (printed)
        mae_validation_fold = mean_absolute_error(y_validation_fold, validation_preds)
        rmse_validation_fold = np.sqrt(mean_squared_error(y_validation_fold, validation_preds))
        r2_validation_fold = r2_score(y_validation_fold, validation_preds)

        print(
            f"Fold {k + 1:02d} | {name} | Validation: "
            f"MAE={mae_validation_fold:.3f}, RMSE={rmse_validation_fold:.3f}, R2={r2_validation_fold:.3f}"
        )

results_nested_df = print_results(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]

[fold RVAE] epoch 0 | loss 1142.566957 | recon 1142.558086 | kl 0.141766
[fold RVAE] epoch 300 | loss 2.236749 | recon 2.066502 | kl 2.723952
[fold RVAE] epoch 600 | loss 2.171079 | recon 2.025943 | kl 2.322171
[fold RVAE] epoch 900 | loss 2.038250 | recon 1.852411 | kl 2.973423
[fold RVAE] epoch 1200 | loss 1.949270 | recon 1.744777 | kl 3.271877
[fold RVAE] epoch 1500 | loss 1.825602 | recon 1.612590 | kl 3.408205
[fold RVAE] epoch 1800 | loss 1.808700 | recon 1.597353 | kl 3.381554
[fold RVAE] epoch 1999 | loss 1.801211 | recon 1.587487 | kl 3.419577
Fold 01 | KNN | Validation: MAE=2.080, RMSE=2.263, R2=0.787
Fold 01 | SVM | Validation: MAE=2.724, RMSE=3.479, R2=0.496
Fold 01 | RF | Validation: MAE=2.944, RMSE=3.534, R2=0.480
Fold 01 | XGBoost | Validation: MAE=2.745, RMSE=3.369, R2=0.527
Fold 01 | MLP | Validation: MAE=2.518, RMSE=3.739, R2=0.417
[fold RVAE] epoch 0 | loss 1154.240368 | recon 1154.231514 | kl 0.141335
[fold RVAE] epoch 300 | loss 2.197540 | recon 2.021645 | kl 2.81

,MAE,RMSE,R2,Pearson r,Pearson p
KNN,1.624516,3.475295,0.604517,0.787801,5.116837e-34
SVM,2.002441,4.095122,0.450866,0.773708,3.891145e-32
RF,1.684323,3.409129,0.619432,0.787605,5.447028e-34
XGBoost,1.903036,3.933013,0.493481,0.715745,1.246810e-25
MLP,2.274790,3.668177,0.559399,0.752250,1.612577e-29


In [17]:
from ci import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,1.624516,3.475295,0.604517,0.787801
ci,"[1.253, 1.986]","[2.835, 3.969]","[0.52, 0.693]","[0.73, 0.851]"


In [6]:
z = pd.read_csv('/mnt/sdb/arafat/stroke_riemann/result_figures/1e-6/zdf_R38.csv') # 2^-1, R=5
z = z.drop(columns=['c']).values
print(z.shape, len(participant_ids))    

(155, 38) 155


In [10]:
# Nested CV: re-train KendallRVAE inside each participant fold

# Set random seeds for reproducibility
import random
import numpy as np
import torch

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from val_test import val_test
from print_results import *
from tqdm.notebook import tqdm


# Regression models with random_state set for reproducibility
models = {'KNN': KNeighborsRegressor()}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_poma)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 3

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)

    Z_train_fold = z[train_idx]
    Z_validation_fold = z[validation_idx]
    Z_test_fold = z[test_idx]

    y_train_fold = y_poma[train_idx]
    y_validation_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold validation metrics (printed)
        mae_validation_fold = mean_absolute_error(y_validation_fold, validation_preds)
        rmse_validation_fold = np.sqrt(mean_squared_error(y_validation_fold, validation_preds))
        r2_validation_fold = r2_score(y_validation_fold, validation_preds)

        print(
            f"Fold {k + 1:02d} | {name} | Validation: "
            f"MAE={mae_validation_fold:.3f}, RMSE={rmse_validation_fold:.3f}, R2={r2_validation_fold:.3f}"
        )

results_nested_df = print_results_regression(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/3 [00:00<?, ?it/s]

Fold 01 | KNN | Validation: MAE=2.320, RMSE=2.926, R2=0.643
Fold 02 | KNN | Validation: MAE=4.320, RMSE=6.280, R2=-2.287
Fold 03 | KNN | Validation: MAE=5.520, RMSE=6.632, R2=-1.139

=== Validation Performance (across all folds) ===
          MAE      RMSE        R2  Pearson r  Pearson p
KNN  4.053333  5.537268 -0.318074   0.526577   0.043733

=== Test Performance (across all folds) ===


,MAE,RMSE,R2,Pearson r,Pearson p
KNN,4.386667,4.936666,0.127403,0.471428,0.076071


In [8]:
from ci import *
ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,1.274839,2.780763,0.746795,0.864715
ci,"[0.969, 1.557]","[2.236, 3.184]","[0.665, 0.82]","[0.822, 0.907]"
